# Initial CIViC investigations

1. [Assertions and evidence](#Assertions-and-evidence)
    1. [Assertions](#Assertions)
    2. [Evidence](#Evidence)
    3. [Summary](#Summary:)
3. [Variants and molecular profiles](#Variants-and-molecular-profiles)
    1. [Variants](#Variants)
    2. [Molecular profiles](#Molecular-profiles)
    3. [Summary](#Summary)

In [84]:
import os
import pandas as pd
from collections import Counter

In [21]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_colwidth', None)

CIViC has nightly and monthly bulk releases in TSV format, and evidence/assertions can either be "accepted" or "submitted" (which includes rejected or pending items). For the following we look at the **most recent monthly release** and only **accepted records**.

The data model is documented [here](https://civic.readthedocs.io/en/latest/model.html).

In [3]:
data_dir = os.environ['DATA_DIR']

In [4]:
!ls $DATA_DIR

01-Oct-2026-AcceptedAssertionSummaries.tsv
01-Oct-2026-AcceptedClinicalEvidenceSummaries.tsv
01-Oct-2026-FeatureSummaries.tsv
01-Oct-2026-MolecularProfileSummaries.tsv
01-Oct-2026-VariantGroupSummaries.tsv
01-Oct-2026-VariantSummaries.tsv


### Assertions and evidence

Each evidence item is associated with a single publication, and assertions are supported by one or more evidence items. However, not every evidence item is associated with an assertion.

[top](#Initial-CIViC-investigations)

#### Assertions

In [7]:
assertions = pd.read_csv(os.path.join(data_dir, '01-Oct-2026-AcceptedAssertionSummaries.tsv'), delimiter='\t')
evidence = pd.read_csv(os.path.join(data_dir, '01-Oct-2026-AcceptedClinicalEvidenceSummaries.tsv'), delimiter='\t')

In [11]:
len(assertions)

149

In [212]:
n_evidence = len(evidence)
n_evidence

4940

In [41]:
evidence_ids_in_assertions = assertions.evidence_item_ids.str.split(',').explode()
len(set(evidence_ids_in_assertions))

594

In [213]:
594/n_evidence

0.12024291497975709

According to the [documentation](https://civic.readthedocs.io/en/latest/model/assertions.html), "An assertion classifies the clinical significance of a variant-disease associate under recognized guidelines." Given this and the numbers, I'm inclined to think we should be looking at the evidence rather than the assertions.

The docs also state "CIViC Assertions are the primary entity that make up CIViC submissions to ClinVar." Assuming this means submit all assertions to ClinVar and only assertions, this would mean we have to look at the evidence to get any more data than is present in ClinVar.

For the rest of this notebook we'll stick to looking at **evidence**.

#### Evidence

[top](#Initial-CIViC-investigations)

In [46]:
evidence.head()

,molecular_profile,molecular_profile_id,disease,doid,phenotypes,therapies,therapy_interaction_type,evidence_type,evidence_direction,evidence_level,significance,evidence_statement,citation_id,source_type,asco_abstract_id,citation,nct_ids,rating,evidence_status,evidence_id,variant_origin,last_review_date,evidence_civic_url,molecular_profile_civic_url,is_flagged
0,JAK2 V617F,64,Lymphoid Leukemia,1037.0,NaN,NaN,NaN,Diagnostic,Supports,B,Negative,"JAK2 V617F is not associated with lymphoid leukemia (B-lineage ALL, T-ALL or CLL).",16081687,PubMed,NaN,"Levine et al., 2005",NaN,4.0,accepted,1,Somatic,2023-01-09 21:46:26 UTC,https://civicdb.org/links/evidence_items/1,https://civicdb.org/links/molecular_profiles/64,False
1,PDGFRA D842V,99,Gastrointestinal Stromal Tumor,9253.0,NaN,NaN,NaN,Diagnostic,Supports,B,Negative,GIST tumors harboring PDGFRA D842V mutation are more likely to be benign than malignant.,15146165,PubMed,NaN,"Lasota et al., 2004",NaN,3.0,accepted,2,Somatic,2023-01-09 21:46:27 UTC,https://civicdb.org/links/evidence_items/2,https://civicdb.org/links/molecular_profiles/99,False
2,DNMT3A R882,32,Acute Myeloid Leukemia,9119.0,NaN,NaN,NaN,Diagnostic,Supports,B,Positive,"Young AML patients (<60 years old) with DNMT3A mutations (60% of which were R882) were older in age, had higher white blood cell counts and had higher platelet counts than patients wildtype for DNMT3A.",22490330,PubMed,NaN,"Ribeiro et al., 2012",NaN,3.0,accepted,4,Somatic,2023-01-09 21:46:25 UTC,https://civicdb.org/links/evidence_items/4,https://civicdb.org/links/molecular_profiles/32,False
3,JAK2 V617F,64,Chronic Myeloid Leukemia,8552.0,NaN,NaN,NaN,Diagnostic,Supports,B,Positive,"JAK2 V617F is associated with myeloid malignanices (AML, MDS, CMML/atypical CML).",16081687,PubMed,NaN,"Levine et al., 2005",NaN,4.0,accepted,5,Somatic,2023-01-09 21:46:26 UTC,https://civicdb.org/links/evidence_items/5,https://civicdb.org/links/molecular_profiles/64,False
4,JAK2 V617F,64,Chronic Myeloid Leukemia,8552.0,NaN,NaN,NaN,Diagnostic,Supports,B,Positive,"JAK2 V617F is associated with myeloid neoplasms (AML, MDS, CMML/aCML).",16081687,PubMed,NaN,"Levine et al., 2005",NaN,4.0,accepted,6,Somatic,2023-01-09 21:46:26 UTC,https://civicdb.org/links/evidence_items/6,https://civicdb.org/links/molecular_profiles/64,False


Evidence has many fields, check the [full documentation](https://civic.readthedocs.io/en/latest/model/evidence/overview.html).

Unlike ClinPGx, the evidence statement is truly free text and not generatable from the other columns.

In [74]:
# "molecular_profile" is the variant or set of variants (see below) - thankfully always present
len(evidence[evidence.molecular_profile.notna()])

4940

In [72]:
# "variant_origin" is the allele origin
Counter(evidence.variant_origin)

Counter({'Somatic': 3665,
         'Rare Germline': 749,
         nan: 385,
         'Common Germline': 64,
         'Unknown': 63,
         'Mixed': 12,
         'Combined': 2})

In [85]:
# "citation_id" is always present
len(evidence[evidence.citation_id.notna()])

4940

In [81]:
# nearly all are PubMed
len(evidence[evidence.source_type != "PubMed"])

22

In [97]:
# American Society of Clinical Oncology (ASCO) or the American Society of Hematology (ASH)
set(evidence[evidence.source_type != "PubMed"].source_type)

{'ASCO', 'ASH'}

In [51]:
# "disease" is the cancer type or subtype
len(evidence[evidence.disease.notna()])

4766

In [214]:
# apparently not a required field, but very common
4766/n_evidence

0.9647773279352226

In [96]:
# disease should always have a DOID
len(evidence[evidence.doid.notna()])

4651

In [64]:
list(set(evidence.disease))[:10]

['Atypical Chronic Myeloid Leukemia, BCR-ABL1 Negative',
 'B-cell Lymphoma',
 'AML With KAT6A::CREBBP Fusion',
 'Transitional Cell Carcinoma',
 'Paraganglioma',
 'Papillary Renal Cell Carcinoma',
 'Astroblastoma, MN1-altered',
 'Appendix Cancer',
 'Pancreatic Endocrine Carcinoma',
 'Granulosa Cell Tumor']

In [66]:
# "phenotypes" is optional and can refer to other phenotypes observed with the variant
len(evidence[evidence.phenotypes.notna()])

944

In [104]:
# comma-separated list, which should correspond to HP terms (though the ID is not present AFAICT)
list(set(evidence.phenotypes))[:10]

['Renal cell carcinoma,Hemangioblastoma,Renal cyst,Retinal capillary hemangioma,Hepatic cysts,Pancreatic cysts',
 'Multiple renal cysts,Renal cyst,Hemangioblastoma',
 'Cerebellar hemangioblastoma,Renal cell carcinoma,Retinal capillary hemangioma',
 'Hematological neoplasm',
 'Retinal capillary hemangioma,Hemangioblastoma,Pheochromocytoma,Renal cell carcinoma',
 'Late young adult onset,Middle age onset,Late onset,Intermediate young adult onset',
 'Renal cell carcinoma,Retinal capillary hemangioma,Hemangioblastoma,Pancreatic cysts',
 'Brain neoplasm',
 'Early young adult onset,Adult onset',
 'Renal cell carcinoma,Sporadic']

In [121]:
# exact relationship between phenotypes and the actual evidence statement is not at all clear or consistent
evidence[evidence.phenotypes.notna()][['disease', 'phenotypes', 'evidence_statement']]

,disease,phenotypes,evidence_statement
6,Acute Leukemia,Infantile onset,Children with early age leukaemia who had second hand smoke exposure are more likely to harbor KRAS mutation.
417,B-lymphoblastic Leukemia/lymphoma,"Pediatric onset,Early young adult onset","In children with bone marrow relapsed B-cell precursor acute lymphoblastic leukemia, those with deletions of IKZF1 had worse event-free and overall survival than patients without deletions (N=68 vs 136, 0.30 vs 0.51, P=0.002; 0.36 vs 0.60, P=0.001)."
1022,Lung Non-small Cell Carcinoma,Brain neoplasm,"This prospective Phase III study named Profile 1014 on NSCLC ALK-rearranged patients assessed intracranial disease control rate (IC-DCR) in the subset of patients entering the study with preexisting treated brain metastasis (tBM). IC-DCR was significantly improved in the tBM group given crizotinib (N=39) over chemotherapy (pemetrexed + cisplatin or carboplatin; N=40) at 12 weeks (85% vs. 45%, P<0.001), and at 24 weeks (56% vs. 25%, P=0.006). Progression free survival was also significantly improved with crizotinib over chemotherapy in the tBM subgroup (9.0 vs. 4.0 months; HR 0.40; P<0.001), where in the intent to treat population it was 10.9 vs. 7.0 months (HR 0.45; P<0.001), indicating that the difficult to treat BM population obtained benefit from crizotinib at both the brain and systemic level."
1128,"B-lymphoblastic Leukemia/lymphoma, BCR-ABL1–like",Juvenile onset,"A 10-year-old boy with B-cell acute lymphoblastic leukemia refractory to conventional chemotherapy and harboring an EBF1-PDGFRB fusion responded to imatinib plus induction chemotherapy (vincristine, daunorubicin, prednisone, and intrathecal methotrexate). Complete remission was achieved with bone marrow MRD of 0.059% by flow cytometry and normal PDGFRB by FISH. Consolidation therapy with cyclophosphamide, cytarabine, mercaptopurine, and imatinib followed and the patient was still in remission 10 months after beginning imatinib."
1223,Melanoma,Brain neoplasm,"In the NCT00880321 trial using mutant BRAF inhibitor dabrafenib in solid tumors, 10 cases of melanoma brain metastases were assessed for dabrafenib response. In nine patients a decrease in size of metastases was seen, and in four cases a complete resolution of brain lesions was observed. This subset of patients had median progression free survival of 4.2 months."
...,...,...,...
4887,NaN,Lung adenocarcinoma,Imielinski et al displays functional evidence (Figure 3A) showing the S214F variant causes increased colony formation and increased pMEK/MEK ratio.
4895,Posterior Fossa Group A Ependymoma,Juvenile onset,"Mariet et al. identified a somatic H3C4 (HIST1H3D) p.Lys28Met (K27M) mutation in a posterior fossa group A ependymoma (EPN_PFA) from an 8-year-old male. The tumor showed H3 K27M immunoreactivity, loss of H3 K27me3, and was clustered within the EPN_PFA-1f DNA methylation subclass (calibrated score 0.99); the patient underwent gross total resection and radiotherapy and was alive at 3 months of follow-up. The authors noted that H3 K27M mutations inhibit PRC2 activity and reduce H3 K27 trimethylation, although no variant-specific functional experiments were performed in this study."
4909,Melanoma,Juvenile onset,"A patient with metastatic melanoma of unknown primary harboring an AGK-BRAF fusion with an intact BRAF kinase domain received belvarafenib 450 mg twice daily after progression on pembrolizumab, dacarbazine, and paclitaxel/carboplatin. Belvarafenib produced a confirmed partial response with a 54.5% reduction in tumor size, and the patient remained progression-free for 16.5 months before progression in pelvic soft-tissue lesions. The fusion preserved BRAF exons 8–18, including the kinase domain."
4938,Urothelial Carcinoma,Bladder carcinoma,"This is a retrospective cohort study examining the association between tumour mutational burden (TMB) and clinical benefit from first-line single-arm immune checkpoint inhibitor (ICPI) therapy compared with carbo

In [99]:
# "evidence_type" is probably the most fundamental classification of the evidence
# https://civic.readthedocs.io/en/latest/model/evidence/type.html
Counter(evidence.evidence_type)

Counter({'Predictive': 2891,
         'Predisposing': 681,
         'Prognostic': 540,
         'Diagnostic': 500,
         'Functional': 174,
         'Oncogenic': 154})

In [91]:
# "significance" - possible values depend on evidence_type
# https://civic.readthedocs.io/en/latest/model/evidence/significance.html
len(evidence[evidence.significance.notna()])

4883

In [95]:
# "direction" - possible values depend on evidence_type and significance
# https://civic.readthedocs.io/en/latest/model/evidence/direction.html
len(evidence[evidence.evidence_direction.notna()])

4923

In [101]:
# "therapies" - only applicable to predictive types (and present in virtually all of them)
len(evidence[evidence.therapies.notna()])

2890

In [108]:
# comma-separated list, which should correspond to NCIT IDs (found in another column)
list(set(evidence.therapies))[:10]

['Capecitabine,Cetuximab,Oxaliplatin',
 'Sunvozertinib',
 'Sunitinib,Regorafenib,Imatinib',
 'MTOR Kinase Inhibitor PP242,Everolimus',
 'Durvalumab',
 'Nelarabine,Arabinosylguanine',
 'Cyproterone Acetate,Flutamide,Nilutamide,Bicalutamide',
 'Peginterferon Alfa-2a',
 'Palbociclib,Fulvestrant',
 'Ionizing Radiotherapy,Cisplatin']

In [111]:
# "evidence_level" refers to the type of study providing the evidence
# https://civic.readthedocs.io/en/latest/model/evidence/level.html
Counter(evidence.evidence_level)

Counter({'C': 1696, 'B': 1643, 'D': 1343, 'A': 225, 'E': 33})

In [114]:
# "rating" relates to curator confidence in the study
# https://civic.readthedocs.io/en/latest/model/evidence/evidence_rating.html
len(evidence[evidence.rating.notna()])

4895

In [117]:
# curious about the correlation between level & rating...
Counter(evidence[evidence.rating.notna()].rating)

Counter({3.0: 2136, 2.0: 1184, 4.0: 1074, 1.0: 299, 5.0: 202})

#### Summary:
* Each evidence item is associated with one publication.
* Clinical assertions aggregate evidence but with a relatively high bar - must have some guideline associated with it.
* The most important field for evidence is the [type](https://civic.readthedocs.io/en/latest/model/evidence/type.html).
* Evidence type determines the interpretation of associated fields like therapies, phenotypes, direction and significance.
* Strength/confidence of evidence is a combination of level (type of study) and rating (numeric confidence value).

[top](#Initial-CIViC-investigations)

### Variants and molecular profiles

Each piece of evidence references a molecular profile, which is generally a single variant but can also be multiple variants connected by boolean operators (AND, OR, NOT).

Each variant is associated with a feature, which is generally a gene but there are a couple other types.

[top](#Initial-CIViC-investigations)

#### Variants

In [32]:
molecular_profiles = pd.read_csv(os.path.join(data_dir, '01-Oct-2026-MolecularProfileSummaries.tsv'), delimiter='\t')
variants = pd.read_csv(os.path.join(data_dir, '01-Oct-2026-VariantSummaries.tsv'), delimiter='\t')
features = pd.read_csv(os.path.join(data_dir, '01-Oct-2026-FeatureSummaries.tsv'), delimiter='\t')

In [215]:
n_variants = len(variants)
n_variants

2014

In [171]:
# "feature_type" - most variants are associated with genes
# https://civic.readthedocs.io/en/latest/model/features/overview.html
Counter(variants.feature_type)

Counter({'Gene': 1751, 'Fusion': 249, 'Factor': 14})

In [216]:
1751/n_variants

0.8694141012909633

In [138]:
# Almost all are on GRCh37 or missing a reference (?!)
Counter(variants.reference_build)

Counter({'GRCh37': 1292, nan: 721, 'GRCh38': 1})

In [131]:
# not many have complete coordinates
len(variants[variants[['chromosome', 'start', 'stop', 'reference_bases', 'variant_bases']].notna().all(axis=1)])

757

In [217]:
757/n_variants

0.3758689175769613

In [165]:
len(variants[variants[['chromosome', 'start', 'stop']].notna().all(axis=1)])

1285

In [218]:
1285/n_variants

0.6380337636544191

In [229]:
# "variant" is the variant name, which is always present but very inconsistent and not unique
len(variants[variants.variant.notna()])

2014

The history saving thread hit an unexpected error (OperationalError('attempt to write a readonly database')).History will not be written to the database.


In [145]:
variants.variant.head(10)

0    Fusion
1     T315I
2     E255K
3      E17K
4    Fusion
5    F1174L
6    R1275Q
7     S214C
8     V600D
9     V600E
Name: variant, dtype: object

In [230]:
# "variant_aliases" can contain rsIDs
variants[variants.variant_aliases.notna()].variant_aliases.head(10)

0        T(9;22)(Q34;Q11),BCR-ABL1,BCR-ABL
1                    THR334ILE,RS121913459
2                        E274K,RS121913448
3                      GLU17LYS,RS34409589
4                                 EML4-ALK
5                   PHE1174LEU,RS863225281
6                   ARG1275GLN,RS113994087
7                   SER214CYS,RS1057519786
8                    RS121913377,VAL600ASP
9    RS113488022,VAL600GLU,V640E,VAL640GLU
Name: variant_aliases, dtype: object

In [161]:
# very approximate number containing RS ID
len(variants[variants.variant_aliases.notna()][variants[variants.variant_aliases.notna()].variant_aliases.str.contains('RS')])

420

In [219]:
420/n_variants

0.20854021847070506

In [186]:
# HGVS and VICC are two other relatively common identifiers, but still not very high coverage even together
variants[variants.hgvs_descriptions.notna()].hgvs_descriptions

1                                                                NM_007313.2:c.1001C>T,NP_005148.2:p.Thr315Ile,ENST00000372348.2:c.1001C>T,NC_000009.11:g.133748283C>T
2                                                                  NC_000009.11:g.133738363G>A,NM_005157.5:c.763G>A,NP_005148.2:p.Glu255Lys,ENST00000318560.5:c.763G>A
3                                                                NC_000014.8:g.105246551C>T,NM_001014432.1:c.49G>A,NP_001014432.1:p.Glu17Lys,ENST00000407796.2:c.49G>A
5                                                                NM_004304.4:c.3522C>A,NP_004295.2:p.Phe1174Leu,NC_000002.11:g.29443695G>T,ENST00000389048.3:c.3522C>A
6                                                                NM_004304.4:c.3824G>A,NP_004295.2:p.Arg1275Gln,NC_000002.11:g.29432664C>T,ENST00000389048.3:c.3824G>A
                                                                                     ...                                                                             

In [178]:
len(variants[variants.hgvs_descriptions.notna()])

862

In [189]:
# VICC fusion nomenclature - https://fusions.cancervariants.org/en/latest/nomenclature.html
variants[variants.vicc_compliant_name.notna()].vicc_compliant_name

0               BCR(entrez:613)::ABL1(entrez:25)
4            EML4(entrez:27436)::ALK(entrez:238)
22        MEF2D(entrez:4209)::CSF1R(entrez:1436)
23      DNAJB1(entrez:3337)::PRKACA(entrez:5566)
42         FGFR2(entrez:2263)::OGA(entrez:10724)
                          ...                   
2005      RUFY1(entrez:80230)::RAF1(entrez:5894)
2006      AGAP3(entrez:116988)::BRAF(entrez:673)
2007    MIPOL1(entrez:145282)::RAF1(entrez:5894)
2009    FGFR2(entrez:2263)::ACOT11(entrez:26027)
2010       CD74(entrez:972)::PDGFRB(entrez:5159)
Name: vicc_compliant_name, Length: 249, dtype: object

In [179]:
len(variants[variants.vicc_compliant_name.notna()])

249

In [191]:
# not sure why but these are always disjoint
len(variants[variants.hgvs_descriptions.notna() | variants.vicc_compliant_name.notna()])

1111

In [190]:
862+249

1111

Thus far it's not at all clear to me how we can reliably identify variants, even ones that are associated with a specific gene and not fusions or factors. Should check how many evidence items are associated with these underspecified variants.

#### Molecular profiles

[top](#Initial-CIViC-investigations)

In [220]:
n_mole_profs = len(molecular_profiles)
n_mole_profs

1992

In [200]:
# not many involve more than a single variant
len(molecular_profiles[molecular_profiles.variant_ids.str.split(',').str.len() > 1])

219

In [221]:
219/n_mole_profs

0.10993975903614457

In [210]:
# most evidence is associated with single variant molecular profiles
single_variant_ids = molecular_profiles[molecular_profiles.variant_ids.str.split(',').str.len() == 1].molecular_profile_id
len(evidence[evidence.molecular_profile_id.isin(single_variant_ids)])

4382

In [223]:
4382/n_evidence

0.8870445344129555

In [228]:
# when molecular profiles involve more than one variant, only the name tells you how to combine them
# for this we of course need to use the variant names rather than the ids...
molecular_profiles[molecular_profiles.variant_ids.str.split(',').str.len() > 1].name

1600                              BRAF V600E AND BRAF V600M
1601                 EML4::ALK Fusion AND ALK Amplification
1602                      BRAF V600E AND BRAF Amplification
1603    BRAF Amplification AND ( BRAF V600E OR BRAF V600K )
1604           EGFR Amplification AND EGFR::SEPTIN14 Fusion
                               ...                         
1970                        EGFR T790M AND EGFR Exon 19 del
1976                               IDH1 R132C OR IDH1 R132H
1977                    HRAS Q61L OR HRAS Q61R OR HRAS G12V
1978                      ALK F1174L AND RANBP2::ALK Fusion
1981                    KRAS G12D AND TSC1 Loss-of-function
Name: name, Length: 219, dtype: object

#### Summary

* Evidence is associated to molecular profiles, which can be one or multiple variants.
* Variants are associated to features, which can be genes, fusions, or other biological factors.
* Most molecular profiles are single variants, and most variants are on genes.
* Identifying variants consistently will be difficult as even single variants on genes don't have coordinates or consistent identifiers.

[top](#Initial-CIViC-investigations)